In [ ]:
subject_id = "170"
# set by the runner (papermill); None means: find settings.toml from cwd
config_path = None
# set by the runner, e.g. a test folder for derivatives_root
path_overrides = None


In [ ]:
from datetime import datetime
from functools import partial

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

from spectral.helpers import load_or_create_report, save_subject_report
from spectral.specparam import (examine_spectra, plot_aperiodic_topomaps,
                                plot_models, specparam2pandas)
from spectral.utils import ProjectPaths, print_timestamp

paths = ProjectPaths(subject_id, config_path=config_path, path_overrides=path_overrides)
subject_id = paths.subject_id  # zero-padded, as in the file names from 01
paths.create_directories()
print_timestamp("Setting up project paths")
paths.show()

# One report per subject: 01 creates it, each step adds its own section.
REPORT_SECTION = "05 Specparam"
report = load_or_create_report(paths, subject_id)
add_fig = partial(report.add_figure, section=REPORT_SECTION, replace=True)

mne.viz.set_browser_backend("matplotlib")
mne.set_config("MNE_BROWSER_THEME", "light")

In [ ]:
from specparam.plts.spectra import plot_spectra
from specparam import SpectralGroupModel


fg = SpectralGroupModel(
    peak_width_limits=[1, 6],
    min_peak_height=0.15,
    peak_threshold=2.0,
    max_n_peaks=6,
    verbose=False,
)

freq_range = [2, 35]

In [ ]:
epochs_interpolated = mne.read_epochs(
    f"{paths.analysis}/sub-{subject_id}_interpolated-epo.fif", preload=True
)

In [ ]:
psd = epochs_interpolated.compute_psd().average()
spectra, freqs = psd.get_data(return_freqs=True)
# Initialize a FOOOFGroup object, with desired settings

# Define the frequency range to fit

with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
    fg.fit(freqs, spectra, freq_range)
fg.plot()

In [ ]:
channel_names = epochs_interpolated.info["ch_names"]
df_channels = pd.DataFrame({"ID": range(len(channel_names)), "ch": channel_names})

df = specparam2pandas(fg)
df = df.merge(df_channels, on="ID")
df["sub_id"] = subject_id


# Get the current date and time
now = datetime.now()
df["timestamp"] = now
#df["nr_intepolated_channels"] = len(epochs_ar.info["bads"])
#df["nr_dropped_ica"] = len(ica.exclude)
#df["nr_retained_ica"] = ica.n_components_ - len(ica.exclude)
# Create a new list of column names
cols = ["ch"] + [col for col in df.columns if col != "ch"]

# Reorder the columns
df = df[cols]

# Plot R and exponent across the scalp

In [ ]:
fg.to_df(None)

In [ ]:
import matplotlib.pyplot as plt

# Offset, exponent and fit quality across the scalp.
fig = plot_aperiodic_topomaps(
    fg, epochs_interpolated.info, subject_id,
    params=("offset", "exponent", "r_squared"),
)
plt.show()
add_fig(fig, title="Offset, Exponent and R_squared values")


In [ ]:
# Compare the power spectra between low and high exponent channels
fig = examine_spectra(fg, subject_id)
plt.show()
add_fig(
    fig, title="Examples of spectra as a function of exponent and R_squared values"
)


In [ ]:
plot_models(fg, param_choice="exponent")


In [ ]:
plot_models(fg, param_choice="r_squared")


In [ ]:
df.to_csv(f"{paths.specparam}/sub-{subject_id}-specparam.csv", index=False)
print(f"Subject {subject_id} done")

In [ ]:
save_subject_report(report, paths, subject_id)